# 00b — Preprocessing 3D
**Dự án:** Latent Manipulation of Brain MRI using Volume-Preserving GANs

**Pipeline:**
```
Raw .nii.gz
    ↓ skull-stripping (ANTsPyNet)
    ├── resample 1mm → normalized/*.nii.gz
    └── giữ nguyên  → unnormalized/*.nii.gz
```

**Output:**
```
preprocessed_3d/
├── normalized/          ← NIfTI đã chuẩn hóa kích thước
├── unnormalized/        ← NIfTI giữ nguyên kích thước gốc
└── preprocessing_log.csv
```

## Bước 1: Cấu hình

In [17]:
import os

# ==== ĐƯỜNG DẪN ====
DATA_ROOT   = '/kaggle/input/datasets/minhbodoi/fgfgbfghhthththt/data'
TSV_PATH    = '/kaggle/input/datasets/minhbodoi/fgfgbfghhthththt/data/participants.tsv'
OUTPUT_ROOT = '/kaggle/working/preprocessed_3d'

# ==== SỐ SUBJECT XỬ LÝ ====
# Đổi thành 'ALL' để xử lý toàn bộ
NUM_SUBJECTS = 10

os.makedirs(f'{OUTPUT_ROOT}/normalized',   exist_ok=True)
os.makedirs(f'{OUTPUT_ROOT}/unnormalized', exist_ok=True)

all_folders = sorted([f for f in os.listdir(DATA_ROOT) if f.startswith('sub-BrainAge')])
print(f'Tổng folder : {len(all_folders)}')

Tổng folder : 12499


## Bước 2: Cài thư viện

In [18]:
!pip install antspyx antspynet nibabel -q

import ants
import antspynet
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f'ANTsPy version: {ants.__version__}')
print('Thư viện sẵn sàng!')

ANTsPy version: 0.6.3
Thư viện sẵn sàng!


## Bước 3: Đọc participants.tsv

In [19]:
df = pd.read_csv(TSV_PATH, sep='\t', low_memory=False)

df_meta = df[['subject_id', 'subject_age', 'subject_sex', 'subject_dx']].copy()
df_meta = df_meta.dropna(subset=['subject_id', 'subject_age'])
df_meta['subject_id']  = df_meta['subject_id'].astype(str).str.strip()
df_meta['subject_age'] = pd.to_numeric(df_meta['subject_age'], errors='coerce')
df_meta = df_meta.dropna(subset=['subject_age'])

age_dict = dict(zip(df_meta['subject_id'], df_meta['subject_age']))
sex_dict = dict(zip(df_meta['subject_id'], df_meta['subject_sex']))
dx_dict  = dict(zip(df_meta['subject_id'], df_meta['subject_dx']))

folders_valid = sorted(list(set(all_folders) & set(age_dict.keys())))
folders = folders_valid if NUM_SUBJECTS == 'ALL' else folders_valid[:NUM_SUBJECTS]

print(f'Subject dùng được : {len(folders_valid)}')
print(f'Sẽ xử lý          : {len(folders)}')
print(f'Tuổi: min={df_meta["subject_age"].min():.1f}, max={df_meta["subject_age"].max():.1f}')

Subject dùng được : 12499
Sẽ xử lý          : 10
Tuổi: min=5.0, max=97.0


## Bước 4: Định nghĩa hàm preprocessing

In [20]:
def find_nii_file(subject_folder_path):
    """Tìm file .nii trong anat/, xử lý cả trường hợp Kaggle giải nén thành folder."""
    anat_path = os.path.join(subject_folder_path, 'anat')
    if not os.path.exists(anat_path):
        return None
    for f in os.listdir(anat_path):
        full_path = os.path.join(anat_path, f)
        if os.path.isfile(full_path) and (f.endswith('.nii.gz') or f.endswith('.nii')):
            return full_path
        if os.path.isdir(full_path) and f.endswith('.nii'):
            for inner_f in os.listdir(full_path):
                inner_path = os.path.join(full_path, inner_f)
                if os.path.isfile(inner_path) and (inner_f.endswith('.nii.gz') or inner_f.endswith('.nii')):
                    return inner_path
    return None


def skull_strip(ants_img):
    """ANTsPyNet brain_extraction — loại bỏ hộp sọ."""
    prob = antspynet.brain_extraction(ants_img, modality='t1')
    mask = ants.threshold_image(prob, 0.5, 1.0, 1, 0)
    return ants_img * mask


def normalize_intensity(ants_img):
    """Chuẩn hóa intensity về [0,1] theo percentile 1-99."""
    data    = ants_img.numpy().astype(np.float32)
    nonzero = data[data > 0]
    if nonzero.size > 0:
        p1, p99 = np.percentile(nonzero, [1, 99])
        data    = (np.clip(data, p1, p99) - p1) / (p99 - p1 + 1e-8)
    return ants.from_numpy(data, origin=ants_img.origin,
                           spacing=ants_img.spacing, direction=ants_img.direction)


def resample_1mm(ants_img):
    """Chuẩn hóa kích thước về 1mm isotropic."""
    return ants.resample_image(ants_img, (1.0, 1.0, 1.0), use_voxels=False, interp_type=1)


print('Hàm preprocessing định nghĩa xong!')

Hàm preprocessing định nghĩa xong!


## Bước 5: Chạy preprocessing

> Đổi `NUM_SUBJECTS = 'ALL'` ở Bước 1 để xử lý toàn bộ dataset.

In [21]:
from tqdm.notebook import tqdm

results = []
errors  = []

print(f'Đang xử lý {len(folders)} subjects...\n')

for subject_id in tqdm(folders, desc='Preprocessing 3D'):
    try:
        nii_path = find_nii_file(os.path.join(DATA_ROOT, subject_id))
        age      = age_dict.get(subject_id, None)

        if nii_path is None:
            errors.append({'subject_id': subject_id, 'error': 'Không tìm thấy .nii'})
            continue
        if age is None:
            errors.append({'subject_id': subject_id, 'error': 'Không có thông tin tuổi'})
            continue

        img   = ants.image_read(nii_path)
        brain = skull_strip(img)

        # UNNORMALIZED — giữ nguyên kích thước
        ants.image_write(
            normalize_intensity(brain),
            f'{OUTPUT_ROOT}/unnormalized/{subject_id}.nii.gz'
        )

        # NORMALIZED — resample 1mm isotropic
        ants.image_write(
            normalize_intensity(resample_1mm(brain)),
            f'{OUTPUT_ROOT}/normalized/{subject_id}.nii.gz'
        )

        results.append({
            'subject_id': subject_id,
            'age'       : age,
            'sex'       : sex_dict.get(subject_id, 'unknown'),
            'dx'        : dx_dict.get(subject_id,  'unknown')
        })

    except Exception as e:
        errors.append({'subject_id': subject_id, 'error': str(e)})

pd.DataFrame(results).to_csv(f'{OUTPUT_ROOT}/preprocessing_log.csv', index=False)

print(f'\n=== HOÀN THÀNH ===')
print(f'Thành công : {len(results)}/{len(folders)}')
if errors:
    print(f'Lỗi        : {len(errors)}')
    for e in errors:
        print(f"  - {e['subject_id']}: {e['error']}")

Đang xử lý 10 subjects...



Preprocessing 3D:   0%|          | 0/10 [00:00<?, ?it/s]


=== HOÀN THÀNH ===
Thành công : 10/10


## Bước 6: Nén và download output

In [22]:
import shutil

shutil.make_archive(
    '/kaggle/working/preprocessed_3d_output',
    'zip',
    '/kaggle/working/preprocessed_3d'
)

size_gb = os.path.getsize('/kaggle/working/preprocessed_3d_output.zip') / 1e9
print(f'Dung lượng: {size_gb:.2f} GB')
print('Xong! Vào Output panel tải file preprocessed_3d_output.zip về.')

Dung lượng: 0.06 GB
Xong! Vào Output panel tải file preprocessed_3d_output.zip về.
